# Threat Intelligence Aggregator & Parser
**Author:** PRADEEPTHI ARE  
**Project:** IBM SkillsBuild Cyber Security BootCamp 2026 (BharatCares)

---

### Step 1: Import Required Libraries & Define Validation Regex

In [1]:
import re
import json
import datetime
import pandas as pd

# Regex pattern definitions for IoC validation
IP_PATTERN = r'^\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b$'
DOMAIN_PATTERN = r'^(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,6}$'
HASH_SHA256_PATTERN = r'^[a-fA-F0-9]{64}$'

print('Libraries imported and validation patterns initialized successfully.')

### Step 2: Ingest Sample Threat Telemetry Data

In [2]:
# Simulated feed input with duplicates and invalid formats
raw_feed_data = [
    {"type": "ip", "value": "192.168.1.100", "source": "Feed_Alpha", "severity": "high"},
    {"type": "ip", "value": "192.168.1.100", "source": "Feed_Beta", "severity": "high"},  # Duplicate
    {"type": "ip", "value": "999.999.999.999", "source": "Feed_Alpha", "severity": "low"}, # Invalid IP
    {"type": "domain", "value": "malicious-example.com", "source": "Feed_Alpha", "severity": "critical"},
    {"type": "domain", "value": "malicious-example.com", "source": "Feed_Gamma", "severity": "critical"}, # Duplicate
    {"type": "hash", "value": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "source": "Feed_Beta", "severity": "medium"}
]

df_raw = pd.DataFrame(raw_feed_data)
print('Raw Telemetry Ingested:')
print(df_raw)

### Step 3: Validate and Deduplicate IoCs

In [3]:
def validate_ioc(ioc_type, value):
    if ioc_type == 'ip':
        return bool(re.match(IP_PATTERN, value))
    elif ioc_type == 'domain':
        return bool(re.match(DOMAIN_PATTERN, value))
    elif ioc_type == 'hash':
        return bool(re.match(HASH_SHA256_PATTERN, value))
    return False

# Filter invalid formats
df_raw['is_valid'] = df_raw.apply(lambda row: validate_ioc(row['type'], row['value']), axis=1)
df_valid = df_raw[df_raw['is_valid'] == True].copy()

# Deduplicate based on IoC value
df_clean = df_valid.drop_duplicates(subset=['value']).drop(columns=['is_valid'])

print('\nSanitized & Deduplicated Threat Intelligence:')
print(df_clean)

### Step 4: Format to Structured STIX-Compliant JSON Output

In [4]:
stix_bundle = {
    "type": "bundle",
    "id": "bundle--" + str(datetime.datetime.now().timestamp()).replace('.', '-'),
    "objects": []
}

for idx, row in df_clean.iterrows():
    indicator = {
        "type": "indicator",
        "spec_version": "2.1",
        "id": f"indicator--{idx+1}",
        "created": datetime.datetime.utcnow().isoformat() + "Z",
        "name": f"{row['type'].upper()} Indicator - {row['value']}",
        "pattern": f"[{row['type']}:value = '{row['value']}']",
        "labels": [row['severity']],
        "description": f"Ingested from {row['source']}"
    }
    stix_bundle["objects"].append(indicator)

print(json.dumps(stix_bundle, indent=2))